# LLM размечают сказки по функциям Проппа · сбор ответов (Groq)

Блокнот отправляет сказки моделям через Groq API и сохраняет ответы в `cache/propp_<MEMBER>.jsonl`. Анализ — в отдельном блокноте `propp_analysis.ipynb`; модели Yandex AI Studio — скрипт `propp_yandex.py`.

**Дизайн.** 3 модели × 3 условия × 11 сказок × 5 итераций, T = 0.7; плюс проверка температуры (T = 0.3) на 3 сказках.
- **A_names** — в промпте номера и названия функций Проппа;
- **B_defs** — номера, названия и короткие определения;
- **C_fewshot** — B_defs + пример: «Гуси-лебеди» (№ 64) по разбору В. Я. Проппа. Сказка 64 в условии C не размечается.

Рядом с блокнотом должны лежать папка `tales/` (тексты, `<номер>.txt`) и `propp_gold.csv` (эталон).

## 1. Установка и импорты

In [ ]:
%pip install -q openai pandas tqdm

In [ ]:
import os, re, json, time, glob, hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

Path("cache").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

## 2. Настройки

Лимиты Groq Free считаются на аккаунт и на модель, поэтому сказки делятся между участниками (`SPLIT`): каждый со своего аккаунта прогоняет все модели на своих сказках. Ответы складываются в один каталог `cache/` для анализа.

In [ ]:
# =============================================================================
GROQ_API_KEY = ""            # https://console.groq.com/keys (или переменная окружения / секрет Colab GROQ_API_KEY)

MEMBER = "member1"           # имя файла кеша: cache/propp_<MEMBER>.jsonl
SPLIT = {                    # сказки делятся между участниками (лимиты Groq — на аккаунт)
    "member1": ["64", "72", "59", "67"],
    "member2": ["137", "87", "79", "70"],
    "member3": ["138", "74", "50"],
}
MY_TALES = SPLIT[MEMBER]
EXCLUDE_TALES = ["116"]      # не помещается в лимит Groq Free (8K TPM) в условиях B/C

N_ITER       = 5
TEMPERATURE  = 0.7
CONDITIONS   = ["A_names", "B_defs", "C_fewshot"]
FEWSHOT_TALE = "64"          # пример в условии C; в условии C эту сказку не размечаем

TEMP_TALES = ["72", "70", "67"]   # проверка температуры
TEMP_LOW   = 0.3                  # ответы пишутся с суффиксом условия _T03
# =============================================================================

MODELS = {
    "gpt-oss-20b":  dict(model_id="openai/gpt-oss-20b",  reasoning_effort="low",  max_tokens=2500),
    "gpt-oss-120b": dict(model_id="openai/gpt-oss-120b", reasoning_effort="low",  max_tokens=2500),
    "qwen3.8-27b":  dict(model_id="qwen/qwen3.8-27b",    reasoning_effort="none", max_tokens=1500),
}
LIMITS = dict(rpm=8, tpm=7000)   # с запасом от Free: 30 RPM, 8K TPM (Qwen: 7K ITPM), 200K TPD на модель

def resolve_key(name, value):
    if value and value.strip():
        return value.strip()
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    v = os.environ.get(name)
    if v:
        return v
    raise RuntimeError(f"Нет ключа {name}: вставь его в ячейку настроек.")

## 3. Данные

In [ ]:
GOLD = pd.read_csv("propp_gold.csv", dtype=str).set_index("tale_id")
GOLD = GOLD.drop(index=[t for t in EXCLUDE_TALES if t in GOLD.index])

def read_tale(tid):
    return Path(f"tales/{tid}.txt").read_text(encoding="utf-8").strip()

TALES = {tid: read_tale(tid) for tid in GOLD.index}

def est_tokens(text):
    # грубо для кириллицы на токенизаторах gpt-oss / qwen: ~1 токен на 2.8 символа
    return int(len(text) / 2.8)

def gold_nums(tid):
    return [int(t) for t in GOLD.loc[tid, "gold_num"].split() if t.isdigit()]   # 'N' отбрасываем

overview = pd.DataFrame({
    "title": GOLD["title"],
    "слов": {t: len(re.findall(r"\w+", TALES[t])) for t in TALES},
    "≈токенов": {t: est_tokens(TALES[t]) for t in TALES},
    "ходов": GOLD["moves"],
    "длина эталона": {t: len(gold_nums(t)) for t in TALES},
}).sort_values("≈токенов")
display(overview)

## 4. Функции и промпты

Модель отвечает номером функции (8–31), а не буквой Проппа: латинская C (противодействие) и кириллическая С (свадьба) выглядят одинаково.

In [ ]:
FUNCS = {
    8:  ("A",  "Вредительство или недостача",
         "антагонист наносит вред кому-то из семьи (похищает, убивает, изгоняет, околдовывает, подменяет, грозит) "
         "ИЛИ кому-то из семьи чего-то не хватает, хочется что-то иметь (невеста, чудесный предмет, лекарство)"),
    9:  ("B",  "Посредничество",
         "о беде или недостаче сообщают; к герою обращаются с просьбой или приказом, его посылают или отпускают"),
    10: ("C",  "Начинающееся противодействие",
         "искатель соглашается или решается действовать против беды"),
    11: ("↑",  "Отправка", "герой покидает дом"),
    12: ("Д",  "Первая функция дарителя",
         "героя испытывают, выспрашивают, просят о помощи или нападают на него — это подготавливает получение "
         "волшебного средства или помощника"),
    13: ("Г",  "Реакция героя",
         "герой реагирует на действия будущего дарителя: выдерживает или не выдерживает испытание, оказывает "
         "услугу, щадит, отказывает"),
    14: ("Z",  "Получение волшебного средства",
         "в распоряжение героя попадает волшебное средство или помощник: дарят, находят, покупают, появляется "
         "сам, предлагает свои услуги"),
    15: ("R",  "Пространственное перемещение",
         "героя переносят, доставляют или приводят к месту, где находится предмет поисков"),
    16: ("Б",  "Борьба", "герой и антагонист вступают в непосредственную борьбу (бой, состязание)"),
    17: ("К",  "Клеймение", "героя метят: рана, кольцо, полотенце, поцелуй"),
    18: ("П",  "Победа", "антагонист побеждён: в бою, в состязании, убит, изгнан"),
    19: ("Л",  "Ликвидация беды или недостачи",
         "исходная беда или недостача устранена: похищенное добыто, убитый оживлён, заколдованный расколдован"),
    20: ("↓",  "Возвращение", "герой возвращается"),
    21: ("Пр", "Преследование", "героя преследуют: гонятся, летят вдогонку, пытаются убить"),
    22: ("Сп", "Спасение от преследования",
         "герой спасается: бегство, препятствия на пути погони, прятание, превращение"),
    23: ("°",  "Неузнанное прибытие", "герой неузнанным прибывает домой или в другую страну"),
    24: ("Ф",  "Притязания ложного героя", "ложный герой предъявляет необоснованные притязания"),
    25: ("З",  "Трудная задача", "герою предлагают трудную задачу"),
    26: ("Р",  "Решение задачи", "задача решена"),
    27: ("У",  "Узнавание", "героя узнают (по метке, кольцу, решённой задаче)"),
    28: ("О",  "Обличение", "ложный герой или антагонист обличён"),
    29: ("Т",  "Трансфигурация", "герою даётся новый облик"),
    30: ("Н",  "Наказание", "враг наказан"),
    31: ("С",  "Свадьба", "герой вступает в брак и/или воцаряется (или получает награду)"),
}
FUNC_IDS = sorted(FUNCS)
SYM = {k: v[0] for k, v in FUNCS.items()}

TASK = ("Ты — фольклорист. Разметь русскую волшебную сказку по функциям действующих лиц "
        "В. Я. Проппа («Морфология сказки», 1928).")

RULES = '''Правила разметки:
1. Используй только функции из списка выше. Записывай их в том порядке, в котором они происходят в тексте.
2. Подготовительную часть сказки (запреты и их нарушения, выведывание, обман до беды) не размечай.
3. Если эпизод повторяется трижды (три брата, три боя, три испытания), отметь его функции один раз.
4. Если сказка состоит из нескольких ходов (новая беда — новый поиск), продолжай разметку: функции могут повторяться.
5. Если одно событие выполняет две функции, запиши обе подряд.
6. К каждой функции приведи короткую цитату из текста (до 8 слов), где она выражена.

Ответ — только JSON без пояснений:
{"functions": [{"f": <номер функции>, "quote": "<цитата>"}, ...]}'''

def func_list(with_defs):
    lines = []
    for k in FUNC_IDS:
        sym, name, d = FUNCS[k]
        lines.append(f"{k} — {name}" + (f": {d}." if with_defs else ""))
    return "\n".join(lines)

SYSTEM = {
    "A_names": TASK + "\n\nФункции (номер — название):\n" + func_list(False) + "\n\n" + RULES,
    "B_defs":  TASK + "\n\nФункции (номер — название: определение):\n" + func_list(True) + "\n\n" + RULES,
}

# ---------- пример для few-shot: «Гуси-лебеди» по разбору Проппа (с. 105–107) ----------
FEWSHOT_ANSWER = [
    (8,  "Налетели гуси-лебеди, подхватили мальчика"),
    (9,  "Пришла девочка, глядь — братца нету!"),
    (10, "бросилась их догонять"),
    (11, "Выбежала в чистое поле"),
    (12, "Печка, печка, скажи, куда гуси полетели?"),
    (13, "О, у моего батюшки пшеничные не едятся!"),
    (12, "попался ёж; хотела она его толкнуть"),
    (13, "побоялась наколоться"),
    (14, "Вон туда-то! — указал."),
    (15, "Вон туда-то! — указал."),
    (19, "Увидела его сестра, подкралась, схватила и унесла"),
    (20, "схватила и унесла"),
    (21, "а гуси за нею в погоню летят"),
    (22, "Речка её посадила под бережок, гуси пролетели"),
]
FEWSHOT_JSON = json.dumps({"functions": [{"f": f, "quote": q} for f, q in FEWSHOT_ANSWER]}, ensure_ascii=False)

FEWSHOT_JSON = json.dumps({"functions": [{"f": f, "quote": q} for f, q in FEWSHOT_ANSWER]}, ensure_ascii=False, separators=(",", ":"))
SYSTEM["C_fewshot"] = (SYSTEM["B_defs"] + "\n\nПример: разметка сказки «Гуси-лебеди» по разбору самого Проппа "
                       "(функции в порядке текста, в quote — фрагмент, где функция выражена):\n" + FEWSHOT_JSON)
print("пример совпадает с эталоном 64:", [f for f, _ in FEWSHOT_ANSWER] == gold_nums(FEWSHOT_TALE))

def skip_combo(cond, tid):
    return cond == "C_fewshot" and tid == FEWSHOT_TALE

def build_messages(cond, tid):
    # system одинаковый для всех сказок → кешируется; 5 итераций одной сказки → кешируется весь промпт
    return [
        {"role": "system", "content": SYSTEM[cond]},
        {"role": "user", "content": "Текст сказки:\n<<<\n" + TALES[tid] + "\n>>>\n\nВерни JSON."},
    ]

for c in CONDITIONS:
    print("=" * 25, c, f"(≈{est_tokens(SYSTEM[c])} ток.)", "=" * 25)
    print(SYSTEM[c][:1500], "...\n")

## 5. Вызов модели, лимиты, кеш, разбор ответа

- 429 (минутный лимит) — ожидание и повтор того же запроса;
- дневной лимит (ожидание > 15 мин) — модель останавливается, перезапуск ячейки продолжает с места остановки;
- 413 (запрос больше минутного лимита) — запись в `results/skipped.csv`;
- пустой ответ или `{"functions": []}` — повтор.

In [ ]:
class DailyLimit(Exception): pass
class TooLarge(Exception): pass
class NoAnswer(Exception): pass

_client = None
def get_client(provider="groq"):
    global _client
    if _client is None:
        from openai import OpenAI
        _client = OpenAI(base_url="https://api.groq.com/openai/v1",
                         api_key=resolve_key("GROQ_API_KEY", GROQ_API_KEY),
                         timeout=180.0, max_retries=0)
    return _client

def check_models():
    have = {m.id for m in get_client().models.list().data}
    for k, c in MODELS.items():
        print(("OK  " if c["model_id"] in have else "НЕТ ") + f"{k:14s} {c['model_id']}")

# ---------- кеш: свой файл на каждого участника, читаем все ----------
CACHE_PATH = Path(f"cache/propp_{MEMBER}.jsonl")

def load_cache():
    cache = {}
    for p in sorted(glob.glob("cache/*.jsonl")):
        for line in Path(p).read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line)
                if rec.get("raw"):
                    cache[rec["key"]] = rec
    return cache

CACHE = load_cache()
print("готовых ответов в кеше:", len(CACHE))

def cache_key(model, cond, it, tid):
    return f"{model}|{cond}|it{it}|{tid}"

def cache_write(key, raw, usage=None):
    rec = {"key": key, "raw": raw, "usage": usage or {}, "ts": time.strftime("%Y-%m-%d %H:%M:%S")}
    CACHE[key] = rec
    with CACHE_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def log_skip(model, cond, it, tid, reason):
    p = Path("results/skipped.csv")
    new = not p.exists()
    with p.open("a", encoding="utf-8") as f:
        if new:
            f.write("model,cond,iter,tale,reason\n")
        f.write(f"{model},{cond},{it},{tid},\"{str(reason)[:200]}\"\n")

# ---------- лимиты ----------
def parse_duration(s):
    # '7.66s' | '2m59.5s' | '1h2m' | '12' → секунды
    if s is None: return None
    s = str(s).strip().lower()
    try: return float(s)
    except ValueError: pass
    tot, ok = 0.0, False
    for m in re.finditer(r"([\d.]+)\s*(ms|h|m|s)", s):
        ok = True
        tot += float(m.group(1)) * {"h": 3600, "m": 60, "s": 1, "ms": 0.001}[m.group(2)]
    return tot if ok else None

def wait_from_error(e):
    resp = getattr(e, "response", None)
    h = getattr(resp, "headers", {}) or {}
    for name in ("retry-after", "x-ratelimit-reset-tokens", "x-ratelimit-reset-requests"):
        d = parse_duration(h.get(name)) if hasattr(h, "get") else None
        if d: return d + 1
    m = re.search(r"try again in ([0-9hms\. ]+)", str(e), re.I)
    if m:
        d = parse_duration(m.group(1))
        if d: return d + 1
    return 30.0

class Gate:
    # скользящее окно токенов за минуту + пауза между запросами
    def __init__(self):
        self.last = defaultdict(float)
        self.used = defaultdict(list)
        self.seen_prompts = defaultdict(dict)   # model -> {hash: время} — для оценки кеша Groq

    def estimate(self, model, messages, max_tokens):
        h = hashlib.md5(json.dumps(messages, ensure_ascii=False).encode()).hexdigest()
        cached = time.time() - self.seen_prompts[model].get(h, 0) < 7000 and MODELS[model]["model_id"].startswith("openai/")
        inp = 0 if cached else sum(est_tokens(m["content"]) for m in messages)
        return h, inp + int(max_tokens * 0.6)

    def before(self, model, est):
        lim = LIMITS
        gap = 60.0 / lim["rpm"] - (time.time() - self.last[model])
        if gap > 0: time.sleep(gap)
        now = time.time()
        self.used[model] = [(t, n) for t, n in self.used[model] if now - t < 60]
        window = sum(n for _, n in self.used[model])
        if self.used[model] and window + est > lim["tpm"]:
            oldest = min(t for t, _ in self.used[model])
            w = max(1.0, 60 - (now - oldest) + 1)
            print(f"  ⏸ {model}: минутный запас токенов ({window}+{est}), жду {w:.0f} с", flush=True)
            time.sleep(w)
        self.last[model] = time.time()

    def after(self, model, h, usage):
        self.seen_prompts[model][h] = time.time()
        if usage:
            spent = usage.get("prompt_tokens", 0) - usage.get("cached_tokens", 0) + usage.get("completion_tokens", 0)
            self.used[model].append((time.time(), int(spent)))

GATE = Gate()

def usage_dict(u):
    if u is None: return {}
    d = {"prompt_tokens": getattr(u, "prompt_tokens", 0) or 0,
         "completion_tokens": getattr(u, "completion_tokens", 0) or 0}
    det = getattr(u, "prompt_tokens_details", None)
    d["cached_tokens"] = (getattr(det, "cached_tokens", 0) or 0) if det else 0
    return d

def make_kwargs(cfg, messages, max_tokens, use_json=True):
    kw = dict(model=cfg["model_id"], messages=messages, temperature=TEMPERATURE, max_tokens=max_tokens)
    if use_json:
        kw["response_format"] = {"type": "json_object"}
    if cfg.get("reasoning_effort"):
        kw["extra_body"] = {"reasoning_effort": cfg["reasoning_effort"]}
    return kw

def call_llm(model, messages):
    from openai import RateLimitError, BadRequestError, APIStatusError, APITimeoutError, APIConnectionError
    cfg = MODELS[model]
    client = get_client()
    max_tokens, use_json, empty_tries, attempt = cfg["max_tokens"], True, 0, 0
    # запрос целиком (вход + max_tokens) должен влезть в 8K TPM, иначе 413
    inp = sum(est_tokens(m["content"]) for m in messages)
    max_tokens = min(max_tokens, max(1200, 7700 - inp))
    while True:
        attempt += 1
        h, est = GATE.estimate(model, messages, max_tokens)
        GATE.before(model, est)
        try:
            r = client.chat.completions.create(**make_kwargs(cfg, messages, max_tokens, use_json))
            u = usage_dict(getattr(r, "usage", None))
            GATE.after(model, h, u)
            text = (r.choices[0].message.content or "").strip()
            if text:
                return text, u
            empty_tries += 1
            if empty_tries >= 2:
                raise NoAnswer(f"пустой ответ (finish_reason={r.choices[0].finish_reason})")
            max_tokens = int(max_tokens * 1.4)
            print(f"  ↺ {model}: пустой ответ, повтор с max_tokens={max_tokens}", flush=True)
        except RateLimitError as e:
            msg = str(e)
            wait = wait_from_error(e)
            daily = re.search(r"per day|TPD|RPD|free-models-per-day", msg, re.I)
            if daily and wait > 900:
                raise DailyLimit(msg[:200])
            print(f"  ⏸ {model}: лимит, жду {wait:.0f} с (попытка {attempt}): {msg[:150]}", flush=True)
            time.sleep(min(wait, 900))
        except BadRequestError as e:
            if use_json:          # некоторые провайдеры не умеют json_object
                use_json = False
                print(f"  ↺ {model}: без response_format: {str(e)[:120]}", flush=True)
                continue
            if attempt > 3: raise
            time.sleep(5)
        except APIStatusError as e:
            code = getattr(e, "status_code", None)
            if code == 413:
                raise TooLarge(str(e)[:200])
            if attempt > 8:
                raise
            time.sleep(min(10 * attempt, 90))
        except (APITimeoutError, APIConnectionError):
            if attempt > 8: raise
            time.sleep(min(10 * attempt, 90))

# ---------- разбор ответа ----------
ROMAN = {"VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12, "XIII": 13, "XIV": 14, "XV": 15, "XVI": 16,
         "XVII": 17, "XVIII": 18, "XIX": 19, "XX": 20, "XXI": 21, "XXII": 22, "XXIII": 23, "XXIV": 24,
         "XXV": 25, "XXVI": 26, "XXVII": 27, "XXVIII": 28, "XXIX": 29, "XXX": 30, "XXXI": 31}

def to_func(v):
    if isinstance(v, (int, float)) and int(v) in FUNCS:
        return int(v)
    s = str(v).strip()
    m = re.match(r"^\s*(\d{1,2})\b", s)
    if m and int(m.group(1)) in FUNCS:
        return int(m.group(1))
    if s.upper() in ROMAN:
        return ROMAN[s.upper()]
    return None

def count_invalid(text):
    # сколько элементов с номером вне 8–31 (модель выдумала функцию) — отбрасываются при разборе
    try:
        d = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return sum(1 for it in d.get("functions", []) if to_func(it.get("f") if isinstance(it, dict) else it) is None)
    except Exception:
        return 0

def parse_answer(text):
    # → список (номер, цитата) или None
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)
    a, b = text.find("{"), text.rfind("}")
    cands = [text[a:b + 1]] if 0 <= a < b else []
    a2, b2 = text.find("["), text.rfind("]")
    if 0 <= a2 < b2:
        cands.append(text[a2:b2 + 1])
    for c in cands:
        try:
            d = json.loads(c)
        except Exception:
            continue
        items = d.get("functions") if isinstance(d, dict) else d
        if not isinstance(items, list):
            continue
        out = []
        for it in items:
            if isinstance(it, dict):
                f = to_func(it.get("f", it.get("function", it.get("id"))))
                q = str(it.get("quote", ""))
            else:
                f, q = to_func(it), ""
            if f is not None:
                out.append((f, q))
        return out
    nums = re.findall(r'"f"\s*:\s*"?(\d{1,2})', text)   # JSON сломан (кавычки внутри цитат) — достаём номера
    if nums:
        return [(int(n), "") for n in nums if int(n) in FUNCS]
    return None

def ask(model, cond, it, tid):
    key = cache_key(model, cond, it, tid)
    if key in CACHE:
        return CACHE[key]["raw"]
    msgs = build_messages(cond, tid)
    raw, u = call_llm(model, msgs)
    for _ in range(2):                 # {"functions": []} — сбой генерации, а не разметка: до 2 повторов
        if parse_answer(raw) != []:
            break
        print(f"  ↺ {key}: пустой список функций, повтор", flush=True)
        raw, u = call_llm(model, msgs)
    if parse_answer(raw) is None:      # одна попытка починить формат
        print(f"  ⚠ {key}: JSON не разобран, прошу переформатировать", flush=True)
        fix = msgs + [{"role": "assistant", "content": raw[:3000]},
                      {"role": "user", "content": 'Верни ТОЛЬКО JSON вида {"functions": [{"f": <номер>, "quote": "<цитата>"}]}.'}]
        raw2, u2 = call_llm(model, fix)
        if parse_answer(raw2) is not None:
            raw, u = raw2, u2
    cache_write(key, raw, u)
    return raw

In [ ]:
check_models()

## 6. Пилот: самая короткая и самая длинная сказка, итерация 1

In [ ]:
PILOT_TALES = [overview.index[0], overview.index[-1]]   # самая короткая и самая длинная
pilot_rows = []
for model in MODELS:
    for cond in CONDITIONS:
        for tid in PILOT_TALES:
            if skip_combo(cond, tid): continue
            try:
                raw = ask(model, cond, 1, tid)
                parsed = parse_answer(raw)
                u = CACHE[cache_key(model, cond, 1, tid)].get("usage", {})
                pilot_rows.append(dict(model=model, cond=cond, tale=tid,
                                       n_funcs=None if parsed is None else len(parsed),
                                       gold_len=len(gold_nums(tid)), **u))
            except (TooLarge, NoAnswer, DailyLimit) as e:
                pilot_rows.append(dict(model=model, cond=cond, tale=tid, error=f"{type(e).__name__}: {e}"))
display(pd.DataFrame(pilot_rows))

In [ ]:
# сырой ответ одной модели — глазами
k = cache_key(next(iter(MODELS)), "B_defs", 1, PILOT_TALES[0])
print(CACHE[k]["raw"][:2000] if k in CACHE else "нет ответа")

## 7. Прогон

### 7.1. Чистка кеша от пустых ответов (если пилот дал `{"functions": []}`)

In [ ]:
def purge_empty(path=CACHE_PATH):
    if not path.exists():
        print("кеша нет"); return
    keep, drop = [], []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        rec = json.loads(line)
        (drop if parse_answer(rec.get("raw", "")) in (None, []) else keep).append(rec)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in keep), encoding="utf-8")
    for r in drop:
        CACHE.pop(r["key"], None)
    print(f"оставлено {len(keep)}, удалено пустых/битых {len(drop)}:", [r["key"] for r in drop])

purge_empty()

### 7.2. Основной прогон: свои сказки `MY_TALES`, T = 0.7

Порядок: модель → сказка → условие → итерации 1–5 подряд. Ячейку можно перезапускать — готовое берётся из кеша.

In [ ]:
def run_all(tales=MY_TALES, models=MODELS, conds=CONDITIONS, n_iter=N_ITER):
    stopped = set()
    combos = [(t, c) for t in tales for c in conds if not skip_combo(c, t)]
    total = len(models) * len(combos) * n_iter
    bar = tqdm(total=total, desc=f"{MEMBER}")
    for model in models:
        for tid, cond in combos:
            for it in range(1, n_iter + 1):
                bar.update(1)
                if model in stopped:
                    continue
                try:
                    ask(model, cond, it, tid)
                except DailyLimit as e:
                    print(f"■ {model}: дневной лимит — на сегодня стоп. {e}", flush=True)
                    stopped.add(model)
                except TooLarge as e:
                    print(f"✗ {model} {cond} {tid}: запрос больше минутного лимита — пропуск", flush=True)
                    log_skip(model, cond, it, tid, e)
                    break                       # остальные итерации тоже не влезут
                except NoAnswer as e:
                    print(f"✗ {model} {cond} it{it} {tid}: {e}", flush=True)
                    log_skip(model, cond, it, tid, e)
    bar.close()
    done = sum(1 for m in models for t, c in combos for i in range(1, n_iter + 1)
               if cache_key(m, c, i, t) in CACHE)
    print(f"готово {done} из {total}" + (f"; остановлены по дневному лимиту: {sorted(stopped)}" if stopped else ""))

run_all()

### 7.3. Проверка температуры: T = 0.3, сказки `TEMP_TALES`, все условия

In [ ]:
def ask_temp(model, cond, it, tid):
    """Тот же запрос, но при TEMP_LOW; ключ кеша — <условие>_T03."""
    global TEMPERATURE
    key = cache_key(model, cond + "_T03", it, tid)
    if key in CACHE:
        return CACHE[key]["raw"]
    old, TEMPERATURE = TEMPERATURE, TEMP_LOW
    try:
        msgs = build_messages(cond, tid)
        raw, u = call_llm(model, msgs)
        for _ in range(2):
            if parse_answer(raw) != []:
                break
            raw, u = call_llm(model, msgs)
    finally:
        TEMPERATURE = old
    cache_write(key, raw, u)
    return raw

for model in MODELS:
    for tid in TEMP_TALES:
        for cond in CONDITIONS:
            if skip_combo(cond, tid):
                continue
            for it in range(1, N_ITER + 1):
                try:
                    ask_temp(model, cond, it, tid)
                except (DailyLimit, TooLarge, NoAnswer) as e:
                    print(f"✗ {model} {cond} {tid} it{it}: {e}")
                    break
print("готово")